# Part 1: UrbanCart paid-channel analysis (Jan–Jun 2026)

All numbers are computed in DuckDB (SQL) straight from the CSVs in `../data/`, and every query is in this notebook. Money is rounded to the nearest ₹, percentages to 1 decimal place, and ratios to 2 decimal places.

**Contents**
0. Setup
1. Data quality: issues found, rows affected, how each was handled
2. Spend, new customers and CAC per paid channel
3. Revenue per customer and repeat purchase rate per channel (including Organic)
4. LTV:CAC per paid channel
5. Monthly CAC by signup month
6. Sanity check
7. Assumptions

## 0. Setup

In [1]:
import duckdb
import pandas as pd

def sql(query):
    """Run a query on the default DuckDB connection and return a DataFrame, which renders as a table."""
    return duckdb.sql(query).df()

# Raw views: the CSVs exactly as delivered. Cleaning happens later, in section 1.5.
duckdb.sql("CREATE OR REPLACE VIEW raw_users  AS SELECT * FROM read_csv('../data/users.csv')")
duckdb.sql("CREATE OR REPLACE VIEW raw_orders AS SELECT * FROM read_csv('../data/orders.csv')")
duckdb.sql("CREATE OR REPLACE VIEW raw_spend  AS SELECT * FROM read_csv('../data/marketing_spend.csv')");

## 1. Data quality

### 1.1 Raw data at a glance

In [2]:
sql("""
SELECT 'users'  AS table_name, COUNT(*) AS n_rows FROM raw_users
UNION ALL SELECT 'orders', COUNT(*) FROM raw_orders
UNION ALL SELECT 'marketing_spend', COUNT(*) FROM raw_spend
""")

,table_name,n_rows
0,users,5135
1,orders,5684
2,marketing_spend,24


In [3]:
# Column types DuckDB inferred: every date and number parsed cleanly (no VARCHAR fallbacks).
pd.concat({t: sql(f"DESCRIBE raw_{t}")[["column_name", "column_type"]] for t in ["users", "orders", "spend"]})

column_name column_type
users  0              user_id      BIGINT
       1          signup_date        DATE
       2                 city     VARCHAR
       3  acquisition_channel     VARCHAR
       4                email     VARCHAR
orders 0             order_id      BIGINT
       1              user_id      BIGINT
       2           order_date        DATE
       3          order_value      BIGINT
       4               status     VARCHAR
spend  0                month     VARCHAR
       1              channel     VARCHAR
       2            spend_inr      BIGINT

### 1.2 Issues found

**Issue 1: duplicate orders.** `order_id` should be unique, but it isn't.

In [4]:
# If distinct_full_rows equals distinct_order_ids, every repeat is an exact copy (same date, value and status)
sql("""
SELECT COUNT(*)                                           AS total_rows,
       COUNT(DISTINCT order_id)                           AS distinct_order_ids,
       (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM raw_orders)) AS distinct_full_rows
FROM raw_orders
""")

,total_rows,distinct_order_ids,distinct_full_rows
0,5684,5478,5478


In [5]:
# How many times each order_id appears
sql("""
SELECT n_copies, COUNT(*) AS n_order_ids
FROM (SELECT order_id, COUNT(*) AS n_copies FROM raw_orders GROUP BY order_id)
GROUP BY n_copies
ORDER BY n_copies
""")

,n_copies,n_order_ids
0,1,5272
1,2,206


**Issue 2: missing acquisition channel.**

In [6]:
sql("""
SELECT COUNT(*) AS users_with_null_channel,
       COUNT(*) FILTER (WHERE user_id IN (SELECT user_id FROM raw_orders WHERE status = 'delivered')) AS of_which_customers
FROM raw_users
WHERE acquisition_channel IS NULL
""")

,users_with_null_channel,of_which_customers
0,72,40


**Issue 3: internal QA/test accounts.** 35 users have `@urbancart.in` emails (`qa.test1` … `qa.test35`). They are all tagged Affiliate and all in Delhi, which real customers wouldn't be.

In [7]:
sql("""
SELECT split_part(email, '@', 2) AS email_domain, COUNT(*) AS n_users
FROM raw_users
GROUP BY email_domain
ORDER BY n_users DESC
""")

,email_domain,n_users
0,gmail.com,1735
1,yahoo.in,1716
2,outlook.com,1649
3,urbancart.in,35


In [8]:
sql("""
SELECT acquisition_channel, city, COUNT(*) AS n_users, MIN(email) AS example_email
FROM raw_users
WHERE email LIKE '%@urbancart.in'
GROUP BY ALL
""")

,acquisition_channel,city,n_users,example_email
0,Affiliate,Delhi,35,qa.test10@urbancart.in


In [9]:
# Test accounts behave nothing like customers: 100% delivered, ~8 orders each, and values up to ₹3,991 vs ₹2,399 max for real customers
sql("""
WITH o AS (SELECT DISTINCT * FROM raw_orders)
SELECT u.email LIKE '%@urbancart.in'                         AS is_test_account,
       COUNT(*)                                              AS orders,
       COUNT(*) FILTER (WHERE status = 'delivered')          AS delivered_orders,
       ROUND(100.0 * COUNT(*) FILTER (WHERE status = 'delivered') / COUNT(*), 1) AS pct_delivered,
       SUM(order_value) FILTER (WHERE status = 'delivered')::BIGINT AS delivered_revenue,
       MAX(order_value)                                      AS max_order_value
FROM o JOIN raw_users u USING (user_id)
GROUP BY is_test_account
ORDER BY is_test_account
""")

,is_test_account,orders,delivered_orders,pct_delivered,delivered_revenue,max_order_value
0,False,5193,4514,86.9,4961884,2399
1,True,285,285,100.0,779790,3991


**Effect of issue 3 on Affiliate.** All the test accounts sit in Affiliate, so leaving them in would flip the channel's verdict:

In [10]:
# Affiliate with vs without test accounts. LTV:CAC = (0.3 * revenue / customers) / (spend / customers) = 0.3 * revenue / spend
sql("""
WITH o AS (SELECT DISTINCT * FROM raw_orders WHERE status = 'delivered'),
cust AS (
    SELECT u.user_id, u.email LIKE '%@urbancart.in' AS is_test, COUNT(*) AS n_orders, SUM(o.order_value) AS revenue
    FROM raw_users u JOIN o USING (user_id)
    WHERE u.acquisition_channel = 'Affiliate'
    GROUP BY ALL
),
spend AS (SELECT SUM(spend_inr) AS spend FROM raw_spend WHERE channel = 'Affiliate'),
versions AS (
    SELECT 'with test accounts' AS version, * FROM cust
    UNION ALL
    SELECT 'test accounts removed', * FROM cust WHERE NOT is_test
)
SELECT version,
       COUNT(*)                                                       AS customers,
       ROUND(MAX(spend) / COUNT(*))::BIGINT                           AS cac,
       ROUND(SUM(revenue) / COUNT(*))::BIGINT                         AS revenue_per_customer,
       ROUND(100.0 * COUNT(*) FILTER (WHERE n_orders >= 2) / COUNT(*), 1) AS repeat_rate_pct,
       ROUND(0.3 * SUM(revenue) / MAX(spend), 2)                      AS ltv_cac
FROM versions, spend
GROUP BY version
ORDER BY version DESC
""")

,version,customers,cac,revenue_per_customer,repeat_rate_pct,ltv_cac
0,with test accounts,540,641,2339,14.1,1.10
1,test accounts removed,505,685,957,8.1,0.42


### 1.3 Checks that passed

None of these needed any cleaning, but each one was checked:

In [11]:
sql("""
SELECT 'users: duplicate user_id'                 AS check_name, COUNT(*) - COUNT(DISTINCT user_id) AS problem_rows FROM raw_users
UNION ALL SELECT 'users: duplicate email',          COUNT(*) - COUNT(DISTINCT lower(trim(email))) FROM raw_users
UNION ALL SELECT 'users: signup outside Jan–Jun',   COUNT(*) FILTER (WHERE signup_date NOT BETWEEN '2026-01-01' AND '2026-06-30') FROM raw_users
UNION ALL SELECT 'users: null/blank city or email', COUNT(*) FILTER (WHERE city IS NULL OR trim(city) = '' OR email IS NULL OR trim(email) = '') FROM raw_users
UNION ALL SELECT 'orders: order outside Jan–Jun',   COUNT(*) FILTER (WHERE order_date NOT BETWEEN '2026-01-01' AND '2026-06-30') FROM raw_orders
UNION ALL SELECT 'orders: null or non-positive value', COUNT(*) FILTER (WHERE order_value IS NULL OR order_value <= 0) FROM raw_orders
UNION ALL SELECT 'orders: status not delivered/cancelled/returned', COUNT(*) FILTER (WHERE status NOT IN ('delivered', 'cancelled', 'returned') OR status IS NULL) FROM raw_orders
UNION ALL SELECT 'orders: user_id not in users',    COUNT(*) FROM raw_orders o WHERE NOT EXISTS (SELECT 1 FROM raw_users u WHERE u.user_id = o.user_id)
UNION ALL SELECT 'orders: placed before user signed up', COUNT(*) FROM raw_orders o JOIN raw_users u USING (user_id) WHERE o.order_date < u.signup_date
UNION ALL SELECT 'spend: not exactly 4 channels x 6 months', ABS(COUNT(*) - 24) + ABS(COUNT(DISTINCT channel || month) - 24) FROM raw_spend
UNION ALL SELECT 'spend: channel name not found in users', COUNT(*) FROM raw_spend WHERE channel NOT IN (SELECT acquisition_channel FROM raw_users WHERE acquisition_channel IS NOT NULL)
""")

,check_name,problem_rows
0,users: duplicate user_id,0
1,users: duplicate email,0
2,users: signup outside Jan–Jun,0
3,users: null/blank city or email,0
4,orders: order outside Jan–Jun,0
5,orders: null or non-positive value,0
6,orders: status not delivered/cancelled/returned,0
7,orders: user_id not in users,0
8,orders: placed before user signed up,0
9,spend: not exactly 4 channels x 6 months,0


In [12]:
# City labels: 8 clean values, no spelling/case variants
sql("""
SELECT city, COUNT(*) AS n_users FROM raw_users GROUP BY city ORDER BY city
""")

,city,n_users
0,Bengaluru,631
1,Chennai,641
2,Delhi,690
3,Hyderabad,653
4,Jaipur,642
5,Kolkata,631
6,Mumbai,633
7,Pune,614


In [13]:
# Channel labels: 5 clean values plus NULL (issue 2)
sql("""
SELECT acquisition_channel, COUNT(*) AS n_users FROM raw_users GROUP BY acquisition_channel ORDER BY n_users DESC
""")

,acquisition_channel,n_users
0,Meta Ads,1300
1,Affiliate,1235
2,Google Ads,1100
3,Organic,828
4,Influencer,600
5,NaN,72


### 1.4 Data quality summary

| # | Table | Issue | Rows affected | How handled | Why |
|---|---|---|---|---|---|
| 1 | orders | 206 `order_id`s appear twice; every repeat is an exact copy of the whole row | 206 extra rows (412 rows involved; 5,684 → 5,478) | Dropped the copies with `SELECT DISTINCT *` | Identical rows are a load/export glitch. Keeping them double-counts revenue and turns some one-time buyers into "repeat" buyers. |
| 2 | users | `acquisition_channel` is NULL | 72 users (40 of them customers) | Kept, labelled `Unknown`, and reported as a separate row. Excluded from paid-channel CAC and LTV:CAC. | We can't tell which channel (if any) paid for them, so guessing would be an assumption either way. At most 40 customers (1.5% of the total) are affected. |
| 3 | users + orders | 35 internal QA/test accounts (`qa.test*@urbancart.in`), all tagged Affiliate | 35 users and their 285 orders (310 raw rows incl. 25 duplicates); all delivered, ₹7,79,790 | Removed the users and all their orders | They aren't customers. Their orders are 100% delivered at implausibly high values. Leaving them in inflates Affiliate's LTV:CAC from 0.42 to 1.10, nearly level with Google Ads. |

Checked and fine: unique user IDs and emails, dates inside Jan–Jun, no missing or negative order values, clean status/city/channel labels, no orphan orders, no orders before signup, spend complete (4 channels × 6 months).

### 1.5 Build the clean tables

Every later section reads only from `users`, `orders` and `spend`.

In [14]:
duckdb.sql("""
CREATE OR REPLACE TABLE users AS
SELECT user_id, signup_date, city, email,
       COALESCE(acquisition_channel, 'Unknown') AS channel      -- issue 2
FROM raw_users
WHERE email NOT LIKE '%@urbancart.in'                           -- issue 3
""")

duckdb.sql("""
CREATE OR REPLACE TABLE orders AS
SELECT DISTINCT *                                               -- issue 1
FROM raw_orders
WHERE user_id IN (SELECT user_id FROM users)                    -- issue 3: drops the test accounts' orders
""")

duckdb.sql("""
CREATE OR REPLACE TABLE spend AS
SELECT month, channel, spend_inr FROM raw_spend
""")

# One row per customer (a user with >= 1 delivered order), used by every metric below.
duckdb.sql("""
CREATE OR REPLACE TABLE customers AS
SELECT u.user_id, u.channel, u.signup_date,
       COUNT(*)           AS delivered_orders,
       SUM(o.order_value) AS delivered_revenue
FROM users u
JOIN orders o USING (user_id)
WHERE o.status = 'delivered'
GROUP BY ALL
""");

In [15]:
# Reconciliation: users 5,135 - 35 test = 5,100. Orders 5,684 - 206 duplicates - 285 test-account orders = 5,193.
sql("""
SELECT 'users'  AS table_name, (SELECT COUNT(*) FROM raw_users)  AS raw_rows, (SELECT COUNT(*) FROM users)  AS clean_rows
UNION ALL SELECT 'orders', (SELECT COUNT(*) FROM raw_orders), (SELECT COUNT(*) FROM orders)
UNION ALL SELECT 'spend',  (SELECT COUNT(*) FROM raw_spend),  (SELECT COUNT(*) FROM spend)
""")

,table_name,raw_rows,clean_rows
0,users,5135,5100
1,orders,5684,5193
2,spend,24,24


## 2. Spend, new customers and CAC per paid channel

**New customer** = a user with ≥1 *delivered* order, counted under their acquisition channel. **CAC** = total Jan–Jun spend ÷ new customers.

In [16]:
sql("""
WITH s AS (SELECT channel, SUM(spend_inr)::BIGINT AS total_spend FROM spend GROUP BY channel),
     signups AS (SELECT channel, COUNT(*) AS signups FROM users GROUP BY channel),
     c AS (SELECT channel, COUNT(*) AS new_customers FROM customers GROUP BY channel)
SELECT channel,
       total_spend,
       signups,
       new_customers,
       ROUND(100.0 * new_customers / signups, 1)       AS signup_to_customer_pct,
       ROUND(total_spend / new_customers)::BIGINT      AS cac
FROM s JOIN signups USING (channel) JOIN c USING (channel)
ORDER BY cac
""")

,channel,total_spend,signups,new_customers,signup_to_customer_pct,cac
0,Google Ads,400000,1100,655,59.5,611
1,Affiliate,346000,1200,505,42.1,685
2,Meta Ads,547000,1300,675,51.9,810
3,Influencer,297000,600,307,51.2,967


## 3. Revenue per customer and repeat purchase rate (all channels)

**Revenue per customer** = delivered revenue Jan–Jun ÷ customers. **Repeat purchase rate** = % of customers with ≥2 delivered orders. Organic and Unknown have no spend, so they get these two metrics but no CAC.

In [17]:
sql("""
SELECT channel,
       COUNT(*)                                                                AS customers,
       SUM(delivered_revenue)::BIGINT                                          AS delivered_revenue,
       ROUND(SUM(delivered_revenue) / COUNT(*))::BIGINT                        AS revenue_per_customer,
       ROUND(100.0 * COUNT(*) FILTER (WHERE delivered_orders >= 2) / COUNT(*), 1) AS repeat_purchase_rate_pct
FROM customers
GROUP BY channel
ORDER BY revenue_per_customer DESC
""")

,channel,customers,delivered_revenue,revenue_per_customer,repeat_purchase_rate_pct
0,Influencer,307,877469,2858,73.0
1,Google Ads,655,1513880,2311,57.7
2,Unknown,40,85089,2127,52.5
3,Organic,461,877167,1903,55.7
4,Meta Ads,675,1124892,1667,44.7
5,Affiliate,505,483387,957,8.1


## 4. LTV:CAC per paid channel

**LTV** = revenue per customer × 30% contribution margin. **LTV:CAC** = LTV ÷ CAC. A ratio below 1.0 means a customer's 6-month contribution doesn't yet cover what it cost to acquire them.

In [18]:
sql("""
WITH s AS (SELECT channel, SUM(spend_inr) AS total_spend FROM spend GROUP BY channel),
     c AS (SELECT channel, COUNT(*) AS customers, SUM(delivered_revenue) AS revenue FROM customers GROUP BY channel)
SELECT channel,
       ROUND(revenue / customers)::BIGINT                       AS revenue_per_customer,
       ROUND(0.30 * revenue / customers)::BIGINT                AS ltv,
       ROUND(total_spend / customers)::BIGINT                   AS cac,
       ROUND((0.30 * revenue / customers) / (total_spend / customers), 2) AS ltv_cac
FROM s JOIN c USING (channel)
ORDER BY ltv_cac DESC
""")

,channel,revenue_per_customer,ltv,cac,ltv_cac
0,Google Ads,2311,693,611,1.14
1,Influencer,2858,857,967,0.89
2,Meta Ads,1667,500,810,0.62
3,Affiliate,957,287,685,0.42


**Best: Google Ads (1.14)**, the only paid channel whose 6-month contribution covers its CAC. **Worst: Affiliate (0.42)**: CAC is mid-range, but its customers spend the least and rarely come back (8.1% repeat rate).

## 5. Monthly CAC by signup month

CAC for month M = spend in month M ÷ new customers who **signed up** in month M (their delivered order can be in M or later). This matches spend to the users it acquired.

In [19]:
monthly = sql("""
WITH c AS (
    SELECT channel, strftime(signup_date, '%Y-%m') AS signup_month, COUNT(*) AS new_customers
    FROM customers
    GROUP BY ALL
)
SELECT s.channel, s.month, s.spend_inr, c.new_customers,
       ROUND(s.spend_inr / c.new_customers)::BIGINT AS cac
FROM spend s
JOIN c ON c.channel = s.channel AND c.signup_month = s.month
ORDER BY s.channel, s.month
""")
monthly

,channel,month,spend_inr,new_customers,cac
0,Affiliate,2026-01,61000,86,709
1,Affiliate,2026-02,56000,79,709
2,Affiliate,2026-03,59000,81,728
3,Affiliate,2026-04,69000,109,633
4,Affiliate,2026-05,56000,84,667
5,Affiliate,2026-06,45000,66,682
6,Google Ads,2026-01,74000,125,592
7,Google Ads,2026-02,59000,95,621
8,Google Ads,2026-03,62000,104,596
9,Google Ads,2026-04,74000,123,602


In [20]:
# CAC per channel by month, plus the Jan -> Jun change
cac_wide = monthly.pivot(index="channel", columns="month", values="cac")
cac_wide["change_jan_to_jun"] = cac_wide["2026-06"] - cac_wide["2026-01"]
cac_wide["change_pct"] = (100 * cac_wide["change_jan_to_jun"] / cac_wide["2026-01"]).round(1)
cac_wide.sort_values("change_pct", ascending=False)

month,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,change_jan_to_jun,change_pct
channel,,,,,,,,
Meta Ads,491,618,789,886,958,1115,624,127.1
Google Ads,592,621,596,602,644,611,19,3.2
Influencer,981,980,913,960,964,1000,19,1.9
Affiliate,709,709,728,633,667,682,-27,-3.8


**Meta Ads got much more expensive: CAC went from ₹491 in January to ₹1,115 in June (+₹624, +127%).** Monthly spend roughly doubled (₹54k → ₹116k), but new customers stayed flat at about 104–123 a month, so the extra money bought almost no extra customers. The other three moved by less than 4% (Google Ads +3.2%, Influencer +1.9%, Affiliate −3.8%).

*Caveat:* June signups had the least time to place a delivered order, so June CAC is slightly overstated for **every** channel. That doesn't explain Meta's rise, which grows steadily from February onwards.

## 6. Sanity check

**Number checked: Meta Ads CAC = ₹810.** Three independent checks:

1. **Recompute it in pandas**, from the raw CSVs, without reusing any SQL.
2. **Reconcile the totals**: per-channel customers and revenue must add up to the totals computed straight from the clean orders table.
3. **Monthly vs total**: Jan–Jun spend ÷ the sum of monthly new customers must equal the overall CAC.

In [21]:
# 1. Independent recompute in pandas
u = pd.read_csv("../data/users.csv")
o = pd.read_csv("../data/orders.csv").drop_duplicates()
s = pd.read_csv("../data/marketing_spend.csv")

u = u[~u["email"].str.endswith("@urbancart.in")]
delivered = o[(o["status"] == "delivered") & o["user_id"].isin(u["user_id"])]
meta_users = u.loc[u["acquisition_channel"] == "Meta Ads", "user_id"]
meta_customers = delivered.loc[delivered["user_id"].isin(meta_users), "user_id"].nunique()
meta_spend = s.loc[s["channel"] == "Meta Ads", "spend_inr"].sum()

pandas_cac = round(meta_spend / meta_customers)
sql_cac = duckdb.sql("SELECT ROUND(SUM(spend_inr) / (SELECT COUNT(*) FROM customers WHERE channel = 'Meta Ads')) FROM spend WHERE channel = 'Meta Ads'").fetchall()[0][0]
print(f"pandas: {meta_spend:,} / {meta_customers} = ₹{pandas_cac}   |   SQL: ₹{sql_cac:.0f}")
assert pandas_cac == sql_cac

pandas: 547,000 / 675 = ₹810   |   SQL: ₹810


In [22]:
sql("""
-- 2. Channel rows add up to independently computed totals
SELECT (SELECT SUM(delivered_revenue)::BIGINT FROM customers)                        AS sum_of_channel_revenue,
       (SELECT SUM(order_value)::BIGINT FROM orders WHERE status = 'delivered')      AS total_delivered_revenue,
       (SELECT COUNT(*) FROM customers)                                              AS sum_of_channel_customers,
       (SELECT COUNT(DISTINCT user_id) FROM orders WHERE status = 'delivered')       AS distinct_delivered_users
""")

,sum_of_channel_revenue,total_delivered_revenue,sum_of_channel_customers,distinct_delivered_users
0,4961884,4961884,2643,2643


In [23]:
# 3. Monthly new customers add up to the Jan-Jun total
meta_monthly = monthly[monthly["channel"] == "Meta Ads"]
print(f"Sum of monthly Meta new customers: {meta_monthly['new_customers'].sum()} (total table: {meta_customers})")
print(f"Spend / customers: ₹{meta_monthly['spend_inr'].sum() / meta_monthly['new_customers'].sum():.0f}")

Sum of monthly Meta new customers: 675 (total table: 675)
Spend / customers: ₹810


All three agree: ₹5,47,000 ÷ 675 Meta customers = **₹810**, the same value from pandas and SQL. Channel revenue and customers reconcile exactly to the totals, and monthly customers sum to the Jan–Jun figure.

## 7. Assumptions

1. **Test accounts** are identified by the `@urbancart.in` domain. All 35 also follow the `qa.test<N>` pattern.
2. **Duplicate orders** are exact copies, so keeping one copy of each loses no information.
3. **NULL channel** users are kept as `Unknown`, not assigned to Organic or to a paid channel. They appear in section 3 but not in CAC.
4. **Channel attribution** is the user's `acquisition_channel` at signup (first-touch). All of a customer's delivered revenue counts towards that channel.
5. **Spend** for a channel buys all of its signups. CAC divides it by the signups that converted, as the brief defines.
6. **Revenue** = delivered orders only. Cancelled and returned orders count as zero revenue, and `order_value` is taken as net revenue to UrbanCart.
7. **LTV** uses observed Jan–Jun revenue only, so it is a 6-month value, not a lifetime one. Later signups had less time to buy again (right-censoring): customers who signed up in June have a 5.8% repeat rate vs ~55% for January–April signups. Channels whose signups shifted later look slightly worse on LTV than they really are.

In [24]:
# Right-censoring evidence for assumption 7
sql("""
SELECT strftime(signup_date, '%Y-%m') AS signup_month,
       COUNT(*) AS customers,
       ROUND(100.0 * COUNT(*) FILTER (WHERE delivered_orders >= 2) / COUNT(*), 1) AS repeat_purchase_rate_pct,
       ROUND(AVG(delivered_revenue))::BIGINT AS revenue_per_customer
FROM customers
GROUP BY signup_month
ORDER BY signup_month
""")

,signup_month,customers,repeat_purchase_rate_pct,revenue_per_customer
0,2026-01,465,55.5,2168
1,2026-02,438,57.8,2222
2,2026-03,402,53.7,2076
3,2026-04,490,54.5,1968
4,2026-05,471,43.9,1626
5,2026-06,377,5.8,1104
